# Aula 10 — Laboratório
## Para quem a política funciona

**Análise Econômica e Machine Learning** · Prof. Dr. Flavius Sodré · Economia, 8º período

Neste laboratório:

1. muitos controles: seleção simples e seleção dupla com o Lasso;
2. o controle ruim que o Lasso guarda, no mundo da Aula 03;
3. o efeito por grupo, à mão, no programa da Aula 09 com efeito que varia;
4. a floresta causal e o teste por quintos;
5. quem chamar com vagas para 30% — e o 401(k) por faixa de renda.

No fim, a **recomendação ao secretário**: com vagas para 30% dos inscritos, quem chamar?

**Como completar:** onde houver `# COMPLETE`, escreva o código. Onde houver `___`,
substitua. Rode as células na ordem.

## Bloco 0 — ambiente

In [ ]:
# Bloco 0 — rode esta celula e confira a saida.
import os, sys, platform, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_rows", 60)
pd.set_option("display.width", 120)
pd.set_option("display.precision", 3)
plt.rcParams["figure.figsize"] = (10, 4.2)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

# As mesmas cores dos slides. Azul = selecao dupla e floresta causal,
# laranja = selecao simples e a regra da necessidade, verde = todas as
# colunas e o sorteio. Bordo = o efeito verdadeiro.
AZUL, LARANJA, VERDE, BORDO, CINZA = "#2a78d6", "#eb6834", "#1baf7a", "#6F1D2B", "#8A8D96"

try:
    import sklearn
    from sklearn.ensemble import (HistGradientBoostingClassifier,
                                  HistGradientBoostingRegressor, RandomForestRegressor)
    from sklearn.linear_model import LassoCV
    from sklearn.model_selection import KFold
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
except ImportError:
    raise ImportError("Este laboratorio precisa do scikit-learn. "
                      "Instale com:  pip install scikit-learn")

try:
    import econml
    TEM_ECONML = True
    versao_econml = econml.__version__
except ImportError:
    TEM_ECONML = False
    versao_econml = "nao instalado"

SEMENTE = 2026
print("Python      :", sys.version.split()[0], "|", platform.system())
print("pandas      :", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("econml      :", versao_econml)
if not TEM_ECONML:
    print("\nPara a floresta causal do Bloco 4, rode numa celula:  %pip install econml")
    print("e depois rode esta celula de novo. Sem ele, o Bloco 4 usa a floresta das")
    print("inclinacoes, so com scikit-learn, e os numeros mudam um pouco.")
print("\nAmbiente pronto.")

In [ ]:
# Bloco 0 — as ferramentas da Aula 09. Leia, nao precisa alterar.

def ols(y, X, nomes):
    """Mínimos quadrados com constante e erro-padrão robusto (HC1).
    Devolve uma tabela com coeficiente e erro-padrão de cada coluna."""
    X = np.column_stack([np.ones(len(y)), np.asarray(X, dtype=float)])
    y = np.asarray(y, dtype=float)
    XtX_inv = np.linalg.pinv(X.T @ X)
    beta = XtX_inv @ X.T @ y
    u = y - X @ beta
    meio = (X * u[:, None]).T @ (X * u[:, None])
    v = XtX_inv @ meio @ XtX_inv * len(y) / (len(y) - X.shape[1])
    return pd.DataFrame({"coef": beta, "ep": np.sqrt(np.diag(v))},
                        index=["constante"] + list(nomes))


def dml(X, y, D, fabrica, dobras=5, semente=SEMENTE):
    """Double/debiased machine learning no modelo parcialmente linear.

    Duas sobras, com dobras cruzadas: a renda que o modelo não previu a partir
    das características e o tratamento que o modelo não previu a partir das
    mesmas características. Cada previsão vem de um modelo ajustado nas outras
    dobras, que não viu aquela linha. O efeito é a inclinação de uma sobra
    contra a outra; o erro-padrão é o robusto dessa regressão."""
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    D = np.asarray(D, dtype=float)
    sobra_y = np.empty(len(y))
    sobra_d = np.empty(len(y))
    for treino, teste in KFold(dobras, shuffle=True, random_state=semente).split(X):
        sobra_y[teste] = y[teste] - fabrica().fit(X[treino], y[treino]).predict(X[teste])
        sobra_d[teste] = D[teste] - fabrica().fit(X[treino], D[treino]).predict(X[teste])
    efeito = float(sobra_d @ sobra_y / (sobra_d @ sobra_d))
    u = sobra_y - efeito * sobra_d
    ep = float(np.sqrt(np.sum((sobra_d * u) ** 2)) / (sobra_d @ sobra_d))
    return {"efeito": efeito, "ep": ep, "sobra_y": sobra_y, "sobra_d": sobra_d}


def modelo_boosting():
    return HistGradientBoostingRegressor(max_iter=300, learning_rate=0.1, max_depth=3,
                                         min_samples_leaf=20, random_state=0)


print("Prontas: ols(), dml() e modelo_boosting().")

## Bloco 1 — muitos controles

Duzentas perguntas no cadastro: quais entram como controle? Na **seleção simples**, o Lasso
escolhe as que preveem a renda. Na **seleção dupla**, escolhe também as que preveem quem faz
o curso, e a regressão usa a união das duas listas.

In [ ]:
# Bloco 1 — o cadastro simulado. Leia, nao precisa alterar.
#
# Inscritos com as respostas padronizadas de 200 perguntas. Cinco perguntas
# preveem a renda; outras cinco levam ao curso e baixam um pouco a renda; o
# resto e ruido. O efeito VERDADEIRO do curso e R$ 300.

N_CADASTRO = 600
N_PERGUNTAS = 200
PREVEEM_RENDA = (0, 5, 15, 25, 35)
PREVEEM_CURSO = (10, 20, 30, 40, 50)
EFEITO_CADASTRO = 300.0


def cadastro(n=N_CADASTRO, p=N_PERGUNTAS, semente=SEMENTE, efeito=EFEITO_CADASTRO):
    """Inscritos com as respostas padronizadas de p perguntas do cadastro.
    Perguntas vizinhas se parecem (correlação 0,5 com a seguinte). Cinco
    perguntas preveem a renda; outras cinco — encaminhado pela assistência
    social, por exemplo — levam ao curso e baixam um pouco a renda."""
    rng = np.random.default_rng(semente)
    Z = rng.normal(size=(n, p))
    X = np.empty_like(Z)
    X[:, 0] = Z[:, 0]
    for j in range(1, p):
        X[:, j] = 0.5 * X[:, j - 1] + np.sqrt(0.75) * Z[:, j]
    z = -0.8 + 0.8 * X[:, list(PREVEEM_CURSO)].sum(axis=1)
    curso = (rng.random(n) < 1 / (1 + np.exp(-z))).astype(float)
    renda = (efeito * curso + 400 * X[:, list(PREVEEM_RENDA)].sum(axis=1)
             - 50 * X[:, list(PREVEEM_CURSO)].sum(axis=1) + rng.normal(0, 600, n))
    colunas = [f"p{j:03d}" for j in range(p)]
    return pd.DataFrame(X, columns=colunas), renda, curso


def lasso_escolhe(X, alvo, semente=0):
    """As colunas que o Lasso mantém para prever o alvo, com o lambda da
    validação cruzada. Alvo e colunas padronizados: a mesma grade serve para
    renda em reais e para curso em 0 e 1."""
    Xp = StandardScaler().fit_transform(np.asarray(X, dtype=float))
    a = np.asarray(alvo, dtype=float)
    z = (a - a.mean()) / a.std()
    m = LassoCV(alphas=np.logspace(-3, 0, 40), max_iter=20000,
                cv=KFold(5, shuffle=True, random_state=semente)).fit(Xp, z)
    return np.flatnonzero(m.coef_ != 0)


def efeito_com(y, D, X, colunas):
    """O coeficiente do tratamento numa regressão com as colunas dadas. As
    colunas entram padronizadas: não muda o coeficiente do tratamento e evita
    a conta com números da ordem de 10^16 (a renda ao cubo)."""
    X = np.asarray(X, dtype=float)[:, list(colunas)]
    X = (X - X.mean(axis=0)) / np.where(X.std(axis=0) > 0, X.std(axis=0), 1)
    t = ols(y, np.column_stack([D, X]), ["d"] + [f"x{c}" for c in colunas])
    return float(t.loc["d", "coef"]), float(t.loc["d", "ep"])


def selecao_simples(X, y, D):
    """O Lasso escolhe os controles olhando só a renda; depois, a regressão."""
    s = lasso_escolhe(X, y)
    return efeito_com(y, D, X, s) + (s,)


perguntas, renda, curso = cadastro()
print(f"{perguntas.shape[0]} inscritos, {perguntas.shape[1]} perguntas; "
      f"{100 * curso.mean():.0f}% fizeram o curso")

In [ ]:
# Bloco 1 — a selecao dupla. COMPLETE as tres lacunas.

def selecao_dupla(X, y, D):
    """O Lasso escolhe olhando a renda e, separado, olhando o tratamento. A
    regressão usa a união das duas listas."""
    # COMPLETE: as duas listas e a uniao delas (np.union1d)
    sy = ___
    sd = ___
    s = ___
    return efeito_com(y, D, X, s) + (s, sy, sd)


print("selecao_dupla() pronta.")

In [ ]:
# Bloco 1 — tres escolhas no cadastro da semente 2026. COMPLETE.

e_s, ep_s, sel_s = selecao_simples(perguntas, renda, curso)
# COMPLETE: a selecao dupla e a regressao com todas as perguntas
e_d, ep_d, sel_d, lista_renda, lista_curso = ___
e_t, ep_t = ___

print(f"simples: {e_s:5.0f} (ep {ep_s:3.0f}) com {len(sel_s)} perguntas")
print(f"dupla  : {e_d:5.0f} (ep {ep_d:3.0f}) com {len(sel_d)} perguntas")
print(f"todas  : {e_t:5.0f} (ep {ep_t:3.0f}) com {perguntas.shape[1]} perguntas")
print("\nperguntas que preveem o curso:", [perguntas.columns[k] for k in PREVEEM_CURSO])
print("guardadas pela simples:", [perguntas.columns[k] for k in sel_s if k in PREVEEM_CURSO])
print("guardadas pela dupla  :", [perguntas.columns[k] for k in sel_d if k in PREVEEM_CURSO])

In [ ]:
# Bloco 1 — cinquenta cadastros. COMPLETE. (Um ou dois minutos.)

linhas = []
for semente in range(50):
    Xc, yc, Dc = cadastro(semente=semente)
    # COMPLETE: o efeito das tres escolhas (so o primeiro elemento de cada resultado)
    linhas.append({"simples": ___,
                   "dupla": ___,
                   "todas": ___})

cinquenta = pd.DataFrame(linhas)
print(cinquenta.describe().loc[["mean", "std", "min", "max"]].round(0))

fig, ax = plt.subplots(figsize=(10, 3.2))
for i, (nome, cor) in enumerate((("dupla", AZUL), ("todas", VERDE), ("simples", LARANJA))):
    ax.scatter(cinquenta[nome], np.full(50, i) + np.random.uniform(-0.15, 0.15, 50),
               color=cor, alpha=0.6, s=30)
ax.axvline(EFEITO_CADASTRO, color=BORDO, lw=2)
ax.set_yticks([0, 1, 2]); ax.set_yticklabels(["dupla", "todas", "simples"])
ax.set_xlabel("efeito estimado (R$ por mes); linha: o verdadeiro")
plt.show()

### Responda

> Quem acerta em média? Quem oscila menos? Por que a seleção simples erra sempre para o
> mesmo lado?

## Bloco 2 — o controle ruim

No mundo da Aula 03, o cargo de chefia é **consequência** do estudo. Ele prevê o salário e
prevê o estudo. O que a seleção dupla faz com ele?

In [ ]:
# Bloco 2 — o mundo da Aula 03. Leia, nao precisa alterar.

EFEITO_DIRETO = 0.06
EFEITO_HABILIDADE = 0.26
EFEITO_EXPERIENCIA = 0.03
EFEITO_EXPERIENCIA2 = -0.0005
EFEITO_CARGO = 0.8
RUIDO = 0.33
CORR_ORIGEM_HAB = 0.92
SETORES = ('Serviços', 'Indústria', 'Comércio', 'Público')
EFEITO_SETOR = {'Serviços': 0.0, 'Indústria': 0.1, 'Comércio': -0.08, 'Público': 0.18}
REGIOES = ('Nordeste', 'Sudeste', 'Sul', 'Norte', 'Centro-Oeste')
EFEITO_REGIAO = {'Nordeste': 0.0, 'Sudeste': 0.2, 'Sul': 0.14, 'Norte': 0.02, 'Centro-Oeste': 0.1}
COLUNAS_A4 = ['experiencia', 'experiencia2', 'origem', 'cargo', 'setor', 'regiao']


def gerar_base(n=4000, semente=42, anos_a_mais=0.0):
    """Uma pesquisa domiciliar fictícia de rendimentos.

    Estrutura de uma PNAD reduzida: uma linha por trabalhador, salário mensal,
    anos de estudo, experiência, origem familiar, setor, região e ocupação de
    chefia. A HABILIDADE existe no mundo e entra no salário, mas NÃO é
    observada — é ela que faz a regressão ingênua mentir.

    `anos_a_mais` faz a intervenção: dá a todo mundo esse tanto de estudo a
    mais, mantendo habilidade, sorteios e ruídos idênticos. É assim que o
    efeito causal verdadeiro é medido, e não lido de um parâmetro.
    """
    rng = np.random.default_rng(semente)

    # habilidade: existe, afeta estudo e salário, e ninguém mede
    hab = rng.normal(0, 1, n)

    # origem familiar (escolaridade dos pais, padronizada): observada, e
    # correlacionada com a habilidade — por isso serve de proxy imperfeita
    origem = (CORR_ORIGEM_HAB * hab
              + np.sqrt(1 - CORR_ORIGEM_HAB ** 2) * rng.normal(0, 1, n))

    # anos de estudo: quem tem mais habilidade e melhor origem estuda mais
    estudo = 9.0 + 1.00 * origem + 1.00 * hab + rng.normal(0, 1.8, n)
    estudo = np.clip(np.round(estudo), 4, 20) + anos_a_mais

    # experiência: anos de trabalho, independente do estudo neste mundo
    experiencia = np.clip(np.round(rng.normal(18, 9, n)), 0, 40)

    setor = rng.choice(SETORES, n, p=[0.45, 0.20, 0.22, 0.13])
    regiao = rng.choice(REGIOES, n, p=[0.30, 0.34, 0.14, 0.08, 0.14])

    # cargo de chefia: CONSEQUÊNCIA do estudo e da habilidade. Controlar por
    # ele fecha parte do caminho pelo qual o estudo aumenta o salário.
    indice = 0.85 * (estudo - 11) / 3 + 0.70 * hab + 0.03 * experiencia
    cargo = (indice + rng.normal(0, 1, n) > 1.10).astype(int)

    log_salario = (
        6.40
        + EFEITO_DIRETO * estudo
        + EFEITO_EXPERIENCIA * experiencia
        + EFEITO_EXPERIENCIA2 * experiencia ** 2
        + EFEITO_HABILIDADE * hab
        + EFEITO_CARGO * cargo
        + np.array([EFEITO_SETOR[s] for s in setor])
        + np.array([EFEITO_REGIAO[r] for r in regiao])
        + rng.normal(0, RUIDO, n)
    )

    return {
        "salario": np.round(np.exp(log_salario), 2),
        "log_salario": log_salario,
        "estudo": estudo,
        "experiencia": experiencia,
        "origem": np.round(origem, 3),
        "setor": setor,
        "regiao": regiao,
        "cargo": cargo,
        "_habilidade": hab,          # não entregar ao aluno
    }


def efeito_causal_verdadeiro(n=200000, semente=2024):
    """Mede o efeito de UM ano a mais de estudo intervindo no mundo.

    Gera a mesma população duas vezes — com os mesmos sorteios de habilidade,
    setor, região e ruído — e na segunda dá um ano de estudo a mais a todos.
    A diferença média no log do salário é o efeito causal total, incluindo a
    parte que passa por virar chefe. Ele NÃO está escrito em nenhuma linha
    do modelo.
    """
    a = gerar_base(n=n, semente=semente, anos_a_mais=0.0)
    b = gerar_base(n=n, semente=semente, anos_a_mais=1.0)
    return float(np.mean(b["log_salario"] - a["log_salario"]))


def tabela_aula03(base=None):
    """O mundo da Aula 03 como tabela, com setor e região em colunas 0/1."""
    b = gerar_base() if base is None else base
    t = pd.DataFrame({k: b[k] for k in ("log_salario", "estudo", "experiencia", "origem",
                                         "cargo", "_habilidade")})
    for s in SETORES[1:]:
        t[f"setor_{s}"] = (b["setor"] == s).astype(float)
    for r in REGIOES[1:]:
        t[f"regiao_{r}"] = (b["regiao"] == r).astype(float)
    return t


def colunas_do_conjunto(t, conjunto):
    cols = []
    for c in conjunto:
        cols += [k for k in t.columns if k == c or k.startswith(c + "_")]
    return cols


def candidatas_aula03(t):
    """As colunas candidatas a controle no mundo da Aula 03: experiência e seu
    quadrado, origem, cargo, setor e região em 0/1."""
    t = t.copy()
    t["experiencia2"] = t["experiencia"] ** 2
    return t[colunas_do_conjunto(t, COLUNAS_A4)]


trabalhadores = tabela_aula03()
candidatas = candidatas_aula03(trabalhadores)
VERDADEIRO_A4 = efeito_causal_verdadeiro()
print(f"{len(trabalhadores)} trabalhadores; {candidatas.shape[1]} candidatas a controle")
print(f"efeito verdadeiro de 1 ano de estudo: {100 * VERDADEIRO_A4:.1f}%")

In [ ]:
# Bloco 2 — a selecao dupla, com e sem o cargo. COMPLETE.

y4, D4 = trabalhadores["log_salario"], trabalhadores["estudo"]
e_com, ep_com, sel_com, sy4, sd4 = selecao_dupla(candidatas, y4, D4)
print("escolhidas:", [candidatas.columns[k] for k in sel_com])
print("o cargo na lista da renda? ", "cargo" in [candidatas.columns[k] for k in sy4])
print("o cargo na lista do estudo?", "cargo" in [candidatas.columns[k] for k in sd4])

# COMPLETE: a selecao dupla sem o cargo entre as candidatas
e_sem, ep_sem, *_ = ___

print(f"\ncom o cargo: {100 * e_com:.1f}%   sem o cargo: {100 * e_sem:.1f}%   "
      f"VERDADEIRO: {100 * VERDADEIRO_A4:.1f}%")

### ✔ CHECKPOINT I

Uma consultoria estima o efeito de um ano de estudo no salário com seleção dupla entre 300
colunas de uma pesquisa domiciliar. Entre as escolhidas estão cargo, setor de atividade e
região de nascimento.

Em duas frases: quais dessas colunas você tiraria antes de rodar, e por quê? O que o Lasso
não tem como saber?

## Bloco 3 — grupos à mão

O programa da Aula 09, com as mesmas regras de inscrição, agora com um efeito que varia de
pessoa para pessoa. As duas sobras são calculadas **uma vez**, para todos; o efeito num
grupo é a inclinação de uma sobra contra a outra **dentro** do grupo.

In [ ]:
# Bloco 3 — o programa da Aula 09, agora com efeito que varia. Leia.

EFEITO_PROGRAMA = 300.0
SALARIO_MINIMO = 1518.0
N_PROGRAMA = 4000
SEMENTE_PROGRAMA = 2026
COLUNAS_PROGRAMA = ['idade', 'escolaridade', 'metropole', 'mulher', 'renda_anterior', 'meses_desempregado']


def efeito_constante(dados):
    """O efeito do programa sobre a renda, pessoa a pessoa: R$ 300 para todos."""
    return np.full(len(dados), EFEITO_PROGRAMA)


def programa(n=N_PROGRAMA, semente=SEMENTE_PROGRAMA, efeito=efeito_constante):
    """Inscritos no sistema público de emprego, com e sem o curso de
    qualificação. A prioridade de vaga vai para quem ganhava menos de R$ 2.200
    (e menos ainda acima de R$ 3.500), para quem tem menos de 30 anos e para
    quem está há mais tempo sem emprego. A renda depois tem um piso: ninguém
    volta ao mercado formal ganhando menos que o salário mínimo.

    Devolve a tabela observada e, à parte, o que só o simulador conhece: a
    probabilidade de participar e o efeito de cada pessoa."""
    rng = np.random.default_rng(semente)
    idade = rng.integers(18, 61, n).astype(float)
    escolaridade = np.clip(np.round(rng.normal(10, 3.2, n)), 2, 17)
    metropole = (rng.random(n) < 0.55).astype(float)
    mulher = (rng.random(n) < 0.5).astype(float)
    a = idade - 18
    log_renda = (6.95 + 0.075 * escolaridade + 0.035 * a - 0.0006 * a ** 2
                 + 0.15 * metropole - 0.12 * mulher + rng.normal(0, 0.32, n))
    renda_anterior = np.round(np.exp(log_renda))
    meses = np.clip(np.round(rng.exponential(4 + 0.15 * a)), 0, 36)

    z = (0.6 - 2.2 * (renda_anterior > 2200) - 0.8 * (renda_anterior > 3500)
         + 0.9 * (idade < 30) + 0.06 * np.minimum(meses, 12) + 0.2 * metropole)
    p = np.clip(1 / (1 + np.exp(-z)), 0.03, 0.97)
    participou = (rng.random(n) < p).astype(float)

    dados = pd.DataFrame({"idade": idade, "escolaridade": escolaridade,
                          "metropole": metropole, "mulher": mulher,
                          "renda_anterior": renda_anterior, "meses_desempregado": meses})
    tau = efeito(dados)
    sem_programa = (np.maximum(SALARIO_MINIMO, 0.92 * renda_anterior)
                    + 0.00004 * np.maximum(renda_anterior - 2000, 0) ** 2
                    - 22 * np.minimum(meses, 24) + 160 * metropole - 120 * mulher
                    + 8 * a - 0.2 * a ** 2)
    ruido = rng.normal(0, 450, n)
    dados["programa"] = participou
    dados["renda_depois"] = np.round(sem_programa + tau * participou + ruido)
    oculto = pd.DataFrame({"prob_participar": p, "efeito": tau,
                           "renda_sem_programa": sem_programa + ruido})
    return dados, oculto


def efeito_heterogeneo(dados):
    """O efeito do curso sobre a renda, pessoa a pessoa (R$ por mês): maior na
    região metropolitana, onde há mais vagas; maior entre os jovens e com mais
    estudo; menor para quem está há muito tempo sem emprego. Nunca negativo."""
    t = (120 + 300 * dados["metropole"] + 220 * (dados["idade"] < 30)
         - 8 * np.minimum(dados["meses_desempregado"], 24)
         + 18 * (dados["escolaridade"] - 10))
    return np.maximum(np.asarray(t, dtype=float), 0)


def efeito_por_grupo(sobra_y, sobra_d, grupo):
    """A inclinação das duas sobras dentro de cada grupo, com erro-padrão: o
    DML da Aula 09, rodado grupo a grupo sobre as sobras de todos."""
    sobra_y, sobra_d, grupo = np.asarray(sobra_y), np.asarray(sobra_d), np.asarray(grupo)
    linhas = {}
    for g in pd.unique(grupo):
        m = grupo == g
        a, b = sobra_d[m], sobra_y[m]
        ef = float(a @ b / (a @ a))
        u = b - ef * a
        linhas[g] = {"efeito": ef, "ep": float(np.sqrt(np.sum((a * u) ** 2)) / (a @ a)),
                     "n": int(m.sum())}
    return pd.DataFrame(linhas).T


def quatro_grupos(dados):
    lugar = np.where(dados["metropole"] == 1, "metrópole", "interior")
    idade = np.where(dados["idade"] < 30, "até 29 anos", "30 anos ou mais")
    return pd.Series(lugar, index=dados.index) + ", " + idade


def subgrupos(dados):
    """48 subgrupos: lugar × 4 faixas de idade × 3 de estudo × sexo."""
    faixa = pd.cut(dados["idade"], [17, 29, 39, 49, 60], labels=["18-29", "30-39", "40-49", "50-60"])
    estudo = pd.cut(dados["escolaridade"], [0, 8, 11, 17], labels=["até 8", "9-11", "12+"])
    lugar = np.where(dados["metropole"] == 1, "metro", "interior")
    sexo = np.where(dados["mulher"] == 1, "M", "H")
    return (pd.Series(lugar, index=dados.index) + "|" + faixa.astype(str) + "|"
            + estudo.astype(str) + "|" + sexo)


inscritos, oculto = programa(efeito=efeito_heterogeneo)
X, y, D = inscritos[COLUNAS_PROGRAMA], inscritos["renda_depois"], inscritos["programa"]
r = dml(X, y, D, modelo_boosting)
sobra_y, sobra_d = r["sobra_y"], r["sobra_d"]
print(f"efeito medio VERDADEIRO: {oculto['efeito'].mean():.0f}; DML: {r['efeito']:.0f}")

In [ ]:
# Bloco 3 — quatro grupos. COMPLETE.

grupo = quatro_grupos(inscritos)
# COMPLETE: a inclinacao das sobras em cada grupo
por_grupo = ___
por_grupo["verdadeiro"] = oculto["efeito"].groupby(grupo).mean()
print(por_grupo.sort_values("verdadeiro").round(0))

In [ ]:
# Bloco 3 — 48 subgrupos. COMPLETE.

sub = subgrupos(inscritos)
por_sub = efeito_por_grupo(sobra_y, sobra_d, sub)
por_sub["verdadeiro"] = oculto["efeito"].groupby(sub).mean()

# COMPLETE: os cinco subgrupos com maior efeito ESTIMADO
cinco = ___
print(cinco.round(0))
print(f"\nmedia dos cinco: estimado {cinco['efeito'].mean():.0f}, "
      f"verdadeiro {cinco['verdadeiro'].mean():.0f}")

### Responda

> Os cinco maiores efeitos estimados valem tanto quanto parecem? Por quê?

## Bloco 4 — floresta causal

A floresta causal procura, sozinha, onde a inclinação das sobras muda. Duas perguntas: ela
acompanha o efeito de cada pessoa? E, quando não há diferença nenhuma, ela inventa?

In [ ]:
# Bloco 4 — a floresta causal. Leia, nao precisa alterar.

def floresta_causal(X, y, D, semente=0):
    """A floresta causal (Wager e Athey, 2018) do pacote econml: árvores que
    cortam os dados onde a inclinação das sobras muda, com as sobras
    calculadas pelo mesmo boosting do DML, em dobras cruzadas."""
    from econml.dml import CausalForestDML
    cf = CausalForestDML(
        model_y=modelo_boosting(),
        model_t=HistGradientBoostingClassifier(max_iter=300, learning_rate=0.1, max_depth=3,
                                               min_samples_leaf=20, random_state=0),
        discrete_treatment=True, n_estimators=500, min_samples_leaf=20, cv=5,
        random_state=semente)
    cf.fit(np.asarray(y, dtype=float), np.asarray(D, dtype=int), X=np.asarray(X, dtype=float))
    return cf


def floresta_das_inclinacoes(X, sobra_y, sobra_d, semente=0):
    """A versão simplificada, só com scikit-learn: uma floresta comum que
    prevê a razão entre as sobras, com peso no quadrado da sobra do
    tratamento. Cada folha estima, aproximadamente, a inclinação das sobras
    das pessoas que caem nela."""
    a = np.asarray(sobra_d, dtype=float)
    f = RandomForestRegressor(n_estimators=300, min_samples_leaf=100, max_features=0.5,
                              n_jobs=-1, random_state=semente)
    return f.fit(np.asarray(X, dtype=float), np.asarray(sobra_y) / a, sample_weight=a ** 2)


def ajustar_e_prever(X, y, D, X_novo):
    """Ajusta a floresta causal em (X, y, D) e prevê o efeito para X_novo.
    Sem o pacote econml, usa a floresta das inclinações sobre as sobras do
    DML."""
    if TEM_ECONML:
        return floresta_causal(X, y, D).effect(np.asarray(X_novo, dtype=float))
    r = dml(X, y, D, modelo_boosting)
    f = floresta_das_inclinacoes(X, r["sobra_y"], r["sobra_d"])
    return f.predict(np.asarray(X_novo, dtype=float))


def fora_da_dobra(X, y, D, semente=SEMENTE):
    """O efeito previsto pela floresta para cada pessoa, por uma floresta que
    não a viu: ajusta numa metade e prevê a outra, e vice-versa."""
    X = np.asarray(X, dtype=float)
    y, D = np.asarray(y, dtype=float), np.asarray(D)
    saida = np.empty(len(X))
    for treino, teste in KFold(2, shuffle=True, random_state=semente).split(X):
        saida[teste] = ajustar_e_prever(X[treino], y[treino], D[treino], X[teste])
    return saida


def teste_por_quintos(previsto, sobra_y, sobra_d):
    """Divide as pessoas em cinco grupos pelo efeito previsto (fora da dobra)
    e estima, em cada um, a inclinação das sobras. Se o efeito previsto
    capta algo de verdade, a inclinação sobe do primeiro ao último quinto."""
    q = pd.qcut(pd.Series(previsto).rank(method="first"), 5, labels=[1, 2, 3, 4, 5])
    t = efeito_por_grupo(sobra_y, sobra_d, q.to_numpy())
    return t.sort_index()


if TEM_ECONML:
    print("Floresta causal: pacote econml.")
else:
    print("ATENCAO: sem o econml, a floresta e a das inclinacoes (so scikit-learn).")

In [ ]:
# Bloco 4 — estimado x verdadeiro. COMPLETE.

# COMPLETE: o efeito previsto para cada inscrito (ajusta e preve em X)
previsto = ___
corr = np.corrcoef(previsto, oculto["efeito"])[0, 1]
print(f"correlacao entre estimado e verdadeiro: {corr:.2f}")

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(oculto["efeito"], previsto, s=6, alpha=0.3, color=AZUL)
ax.plot([0, 800], [0, 800], "--", color=CINZA)
ax.set_xlabel("efeito verdadeiro (R$)"); ax.set_ylabel("efeito estimado (R$)")
plt.show()

In [ ]:
# Bloco 4 — o teste por quintos, nos dois mundos. COMPLETE. (Uns 30 segundos.)

constante, oculto_c = programa()             # o mundo da Aula 09: R$ 300 para todos
rc = dml(constante[COLUNAS_PROGRAMA], constante["renda_depois"], constante["programa"],
         modelo_boosting)

testes = {}
for nome, dados, sy, sd in (("efeito varia", inscritos, sobra_y, sobra_d),
                            ("efeito constante", constante, rc["sobra_y"], rc["sobra_d"])):
    # COMPLETE: o efeito previsto FORA DA DOBRA e o teste por quintos
    prev = ___
    testes[nome] = ___
    print(f"\n{nome}: dispersao do previsto {np.std(prev):.0f}")
    print(testes[nome].round(0))

## Bloco 5 — quem chamar

Vagas para 30% dos inscritos. Três regras: chamar quem a floresta diz que ganha mais; chamar
quem teria a menor renda sem o curso (a necessidade); sortear. Como o mundo é simulado, o
ganho de cada regra é medido com o efeito **verdadeiro**.

In [ ]:
# Bloco 5 — as regras de quem tratar. Leia, nao precisa alterar.

FRACOES = np.round(np.linspace(0, 1, 41), 3)
VAGAS = 0.30


def renda_sem_curso_prevista(X, y, D, semente=SEMENTE):
    """A renda que cada inscrito teria sem o curso, prevista por um boosting
    ajustado só em quem não fez, em dobras cruzadas: a régua da necessidade."""
    X = np.asarray(X, dtype=float)
    y, D = np.asarray(y, dtype=float), np.asarray(D)
    saida = np.empty(len(y))
    for treino, teste in KFold(5, shuffle=True, random_state=semente).split(X):
        t0 = treino[D[treino] == 0]
        saida[teste] = modelo_boosting().fit(X[t0], y[t0]).predict(X[teste])
    return saida


def ganho_por_regra(prioridade, tau, fracoes=FRACOES):
    """Renda a mais por mês, somada, para cada 1.000 inscritos, quando o curso
    vai para a fração dos inscritos com maior prioridade."""
    ordem = np.argsort(-np.asarray(prioridade), kind="stable")
    acum = np.concatenate([[0], np.cumsum(np.asarray(tau)[ordem])])
    n = len(tau)
    return np.array([acum[int(round(f * n))] / n * 1000 for f in fracoes])


print("Prontas: renda_sem_curso_prevista() e ganho_por_regra().")

In [ ]:
# Bloco 5 — tres regras. COMPLETE.

prev_fora = fora_da_dobra(X, y, D)                       # floresta, fora da dobra
necessidade = renda_sem_curso_prevista(X, y, D)          # menor = mais necessitado
tau = oculto["efeito"].to_numpy()

# COMPLETE: o ganho de cada regra; maior prioridade = chamado primeiro
curvas = {
    "floresta": ___,
    "necessidade": ___,
    "sorteio": FRACOES * tau.mean() * 1000,
}
i30 = int(np.argmin(np.abs(FRACOES - VAGAS)))
print({k: round(v[i30] / 1000) for k, v in curvas.items()}, "(R$ mil por 1.000 inscritos)")

n = int(VAGAS * len(tau))
perfil = pd.DataFrame({
    "floresta": inscritos.iloc[np.argsort(-prev_fora)[:n]].mean(),
    "necessidade": inscritos.iloc[np.argsort(necessidade)[:n]].mean(),
    "todos": inscritos.mean()}).T
print(perfil[["idade", "escolaridade", "metropole", "renda_anterior",
              "meses_desempregado"]].round(1))

In [ ]:
# Bloco 5 — o 401(k), o mesmo da Aula 09. Rode e siga.
#
# Tres caminhos: o arquivo Dados/401k.csv do repositorio; o arquivo
# original no repositorio de replicacao do DML; o pacote doubleml. Se os tres
# falharem, segue com familias INVENTADAS (Plano B), com aviso.

COLUNAS_401K = ['nifa', 'net_tfa', 'tw', 'age', 'inc', 'fsize', 'educ', 'db', 'marr', 'twoearn', 'e401', 'p401', 'pira', 'hown']
CONTROLES = ['age', 'inc', 'educ', 'fsize', 'marr', 'twoearn', 'db', 'pira', 'hown']
URL = "https://github.com/VC2015/DMLonGitHub/raw/master/sipp1991.dta"
ESP_401K = {'age': {25: 0.0235,
         26: 0.0271,
         27: 0.0302,
         28: 0.0314,
         29: 0.0319,
         30: 0.0348,
         31: 0.0378,
         32: 0.0344,
         33: 0.0338,
         34: 0.0348,
         35: 0.0341,
         36: 0.0358,
         37: 0.033,
         38: 0.0359,
         39: 0.0307,
         40: 0.0337,
         41: 0.0308,
         42: 0.0345,
         43: 0.0331,
         44: 0.0301,
         45: 0.0255,
         46: 0.0281,
         47: 0.0225,
         48: 0.024,
         49: 0.0215,
         50: 0.0187,
         51: 0.0174,
         52: 0.0205,
         53: 0.0187,
         54: 0.0186,
         55: 0.0167,
         56: 0.0156,
         57: 0.0133,
         58: 0.0124,
         59: 0.0149,
         60: 0.0145,
         61: 0.0144,
         62: 0.0112,
         63: 0.0103,
         64: 0.01},
 'educ': {1: 0.0045,
          2: 0.001,
          3: 0.0027,
          4: 0.0034,
          5: 0.0034,
          6: 0.0125,
          7: 0.005,
          8: 0.016,
          9: 0.0179,
          10: 0.0309,
          11: 0.0309,
          12: 0.3768,
          13: 0.0856,
          14: 0.1137,
          15: 0.0452,
          16: 0.1329,
          17: 0.0325,
          18: 0.085},
 'fsize': {1: 0.2269,
           2: 0.2347,
           3: 0.1955,
           4: 0.2084,
           5: 0.0881,
           6: 0.0289,
           7: 0.0106,
           8: 0.0041,
           9: 0.0009,
           10: 0.0009,
           11: 0.0003,
           12: 0.0004,
           13: 0.0002},
 'log_renda': [7.489392, 0.09284, 0.068997, -0.000695],
 'dp_log_renda': 0.6548,
 'marr': [0.0467, 1.4539],
 'db': [-1.2115, 0.599],
 'pira': [-1.8491, 1.5074],
 'hown': [0.2606, 1.18],
 'e401': [-0.9682, 1.1656],
 'twoearn_dado_casado': 0.6291,
 'p401_dado_elegivel': 0.7045,
 'patrimonio': [-19383.7552, -0.5774, 612.1214, 219.8488, -1292.2897, 5392.0038, -14553.4256,
                -3230.6675, 30636.1176, 4638.4365, 9474.2032, 11.941],
 'dp_patrimonio': 54223.0}


def base_401k_sintetica(esp, n=9915, semente=0):
    """Famílias INVENTADAS com as mesmas colunas e tipos da base real: as
    frequências de idade, estudo e família, a renda pela idade e pelo estudo,
    as demais características e a elegibilidade pela renda, e o patrimônio por
    uma regressão linear com ruído de cauda pesada."""
    rng = np.random.default_rng(semente)

    def sorteia(freq):
        k = np.array([int(x) for x in freq])
        p = np.array(list(freq.values()), dtype=float)
        return rng.choice(k, n, p=p / p.sum())

    age, educ, fsize = sorteia(esp["age"]), sorteia(esp["educ"]), sorteia(esp["fsize"])
    b = esp["log_renda"]
    inc = np.round(np.exp(b[0] + b[1] * educ + b[2] * age + b[3] * age ** 2
                          + rng.normal(0, esp["dp_log_renda"], n)))
    inc = np.minimum(inc, 250000)
    lr = np.log(np.maximum(inc, 1000)) - 10

    def sim_nao(w):
        return (rng.random(n) < 1 / (1 + np.exp(-(w[0] + w[1] * lr)))).astype(int)

    marr, db, pira, hown, e401 = (sim_nao(esp[c]) for c in ("marr", "db", "pira", "hown", "e401"))
    twoearn = marr * (rng.random(n) < esp["twoearn_dado_casado"]).astype(int)
    p401 = e401 * (rng.random(n) < esp["p401_dado_elegivel"]).astype(int)
    X = np.column_stack([np.ones(n), inc, age, educ, fsize, marr, twoearn, db, pira, hown, e401,
                         (inc / 1000.0) ** 2])
    ruido = rng.standard_t(3, n) * esp["dp_patrimonio"] / np.sqrt(3)
    net_tfa = np.round(X @ np.array(esp["patrimonio"]) + ruido)
    nifa = np.maximum(np.round(net_tfa + rng.normal(0, 3000, n)), 0)
    tw = np.round(net_tfa + hown * rng.gamma(2.0, 30000, n))
    return pd.DataFrame({"nifa": nifa, "net_tfa": net_tfa, "tw": tw, "age": age, "inc": inc,
                         "fsize": fsize, "educ": educ, "db": db, "marr": marr,
                         "twoearn": twoearn, "e401": e401, "p401": p401, "pira": pira,
                         "hown": hown}).astype(int)


def modelo_floresta():
    return RandomForestRegressor(n_estimators=300, min_samples_leaf=5, max_features=0.5,
                                 n_jobs=-1, random_state=0)


def carregar_401k(arquivo="../Dados/401k.csv"):
    if os.path.exists(arquivo):
        return pd.read_csv(arquivo)[COLUNAS_401K], "arquivo local " + arquivo
    try:
        return pd.read_stata(URL)[COLUNAS_401K].round().astype(int), "repositorio do DML"
    except Exception as erro:
        print("download falhou:", type(erro).__name__, str(erro)[:80])
    try:
        from doubleml.datasets import fetch_401K
        return fetch_401K(return_type="DataFrame")[COLUNAS_401K], "pacote doubleml"
    except Exception:
        pass
    print("=" * 70)
    print("PLANO B: os dados abaixo sao INVENTADOS. Os numeros NAO valem como")
    print("resultado sobre o 401(k).")
    print("=" * 70)
    return base_401k_sintetica(ESP_401K), "PLANO B (sintetico)"


familias, FONTE = carregar_401k()
print(f"fonte: {FONTE}; {len(familias)} familias")

In [ ]:
# Bloco 5 — o 401(k) por quinto de renda. COMPLETE.

r401 = dml(familias[CONTROLES], familias["net_tfa"], familias["e401"], modelo_floresta)
quinto = pd.qcut(familias["inc"], 5, labels=[1, 2, 3, 4, 5]).to_numpy()
# COMPLETE: a inclinacao das sobras em cada quinto de renda
por_quinto = ___
print(por_quinto.sort_index().round(0))

### ✔ CHECKPOINT II

Com vagas para 30%, a floresta chama sobretudo jovens da metrópole e rende cerca de R$ 150
mil por mês para cada 1.000 inscritos; chamar os mais necessitados rende cerca de R$ 75 mil.

Em duas frases: que regra você recomendaria ao secretário, e por quê? Que informação, fora
do modelo, mudaria a sua resposta?

## A recomendação ao secretário

Em até cinco frases: com vagas para 30% dos inscritos, quem chamar? Cite a estimativa que
sustenta a escolha, diga o que ela deixa de fora e como você verificaria, depois, se a regra
funcionou.

_(escreva aqui)_